# QUY TRÌNH 2 - DATA INGESTION

## E-commerce Data Lakehouse trên Object Storage

Mục tiêu:

- Đọc dữ liệu CSV Raw từ Quy trình 1
- Đọc dữ liệu ảnh sản phẩm từ thư mục images
- Kiểm tra metadata của dữ liệu
- Kết nối đến MinIO
- Upload dữ liệu bảng vào Object Storage
- Upload dữ liệu ảnh vào Object Storage
- Ghi log quá trình ingestion
- Kiểm tra kết quả ingestion

In [1]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
from minio import Minio

In [2]:
# Xác định thư mục gốc của project

BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
IMAGE_DIR = BASE_DIR / "data" / "images" / "products"

# Kiểm tra thư mục CSV local raw
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy local raw directory: {RAW_DIR}"
    )

# Kiểm tra thư mục ảnh
if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy image directory: {IMAGE_DIR}"
    )

print("Project root :", BASE_DIR)
print("Local raw    :", RAW_DIR)
print("Image data   :", IMAGE_DIR)

Project root : D:\E-commerce-Data-Lakehouse
Local raw    : D:\E-commerce-Data-Lakehouse\data\raw
Image data   : D:\E-commerce-Data-Lakehouse\data\images\products


In [3]:
# Load biến môi trường từ file .env

ENV_FILE = BASE_DIR / ".env"

if not ENV_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file .env: {ENV_FILE}"
    )

load_dotenv(ENV_FILE)

MINIO_ENDPOINT = os.getenv(
    "MINIO_ENDPOINT",
    "localhost:9000"
)

MINIO_ACCESS_KEY = os.getenv(
    "MINIO_ROOT_USER"
)

MINIO_SECRET_KEY = os.getenv(
    "MINIO_ROOT_PASSWORD"
)

if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise EnvironmentError(
        "Thiếu MINIO_ROOT_USER hoặc "
        "MINIO_ROOT_PASSWORD trong .env"
    )

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

print("✓ Đã khởi tạo MinIO client")
print("Endpoint:", MINIO_ENDPOINT)

✓ Đã khởi tạo MinIO client
Endpoint: localhost:9000


In [4]:
# ID đại diện cho toàn bộ một lần chạy ingestion

RUN_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

RUN_TIMESTAMP = datetime.now(
    timezone.utc
).isoformat()

print("Run ID:", RUN_ID)
print("UTC time:", RUN_TIMESTAMP)

Run ID: 20260930T153809Z
UTC time: 2026-09-30T15:38:09.555611+00:00


In [5]:
try:
    buckets = list(
        client.list_buckets()
    )

    print("✓ Kết nối MinIO thành công")

    if buckets:
        print("\nCác bucket hiện có:")

        for bucket in buckets:
            print("-", bucket.name)

    else:
        print("Chưa có bucket nào")

except Exception as e:
    raise ConnectionError(
        f"Không thể kết nối MinIO tại "
        f"{MINIO_ENDPOINT}: {e}"
    )

✓ Kết nối MinIO thành công

Các bucket hiện có:
- ecommerce-lakehouse


In [6]:
BUCKET_NAME = "ecommerce-lakehouse"

if not client.bucket_exists(
    BUCKET_NAME
):
    client.make_bucket(
        BUCKET_NAME
    )

    print(
        f"✓ Đã tạo bucket: "
        f"{BUCKET_NAME}"
    )

else:
    print(
        f"✓ Bucket đã tồn tại: "
        f"{BUCKET_NAME}"
    )

✓ Bucket đã tồn tại: ecommerce-lakehouse


In [7]:
DATASETS = {
    "customers":
        RAW_DIR / "customers.csv",

    "products":
        RAW_DIR / "products.csv",

    "orders":
        RAW_DIR / "orders.csv",

    "order_items":
        RAW_DIR / "order_items.csv",

    "reviews":
        RAW_DIR / "reviews.csv",

    "payments":
        RAW_DIR / "payments.csv"
}

missing_files = [
    str(path)
    for path in DATASETS.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Thiếu các CSV local: "
        f"{missing_files}"
    )

print(
    f"✓ Tìm thấy "
    f"{len(DATASETS)} dataset CSV"
)

✓ Tìm thấy 6 dataset CSV


In [8]:
# Dataset demo có kích thước nhỏ nên dùng Pandas
# để lấy metadata/schema.
#
# Với dữ liệu lớn, ingestion có thể upload file
# trực tiếp lên Object Storage mà không cần đọc
# toàn bộ file vào memory.

datasets = {}

for name, path in DATASETS.items():

    df = pd.read_csv(path)

    datasets[name] = df

    print(
        f"✓ {name}: "
        f"{len(df):,} rows, "
        f"{len(df.columns)} columns"
    )

✓ customers: 99,441 rows, 5 columns
✓ products: 32,951 rows, 9 columns
✓ orders: 99,441 rows, 8 columns
✓ order_items: 112,650 rows, 7 columns
✓ reviews: 99,224 rows, 7 columns
✓ payments: 103,886 rows, 5 columns


In [9]:
SUPPORTED_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

IMAGE_FILES = sorted([
    file
    for file in IMAGE_DIR.iterdir()
    if (
        file.is_file()
        and file.suffix.lower()
        in SUPPORTED_IMAGE_EXTENSIONS
    )
])

print(
    f"✓ Tìm thấy "
    f"{len(IMAGE_FILES):,} ảnh sản phẩm"
)

✓ Tìm thấy 5 ảnh sản phẩm


In [10]:
def get_file_metadata(
    file_path
):

    stat = file_path.stat()

    return {
        "file_name":
            file_path.name,

        "file_type":
            file_path.suffix.lower(),

        "file_size_bytes":
            stat.st_size,

        "file_size_mb":
            round(
                stat.st_size
                / (1024 * 1024),
                2
            ),

        "modified_time":
            datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc
            ).isoformat()
    }


metadata = []

for name, path in DATASETS.items():

    info = get_file_metadata(
        path
    )

    info["dataset"] = name

    info["rows"] = len(
        datasets[name]
    )

    info["columns"] = len(
        datasets[name].columns
    )

    metadata.append(
        info
    )


metadata_df = pd.DataFrame(
    metadata
)

metadata_df

,file_name,file_type,file_size_bytes,file_size_mb,modified_time,dataset,rows,columns
0,customers.csv,.csv,8661712,8.26,2026-09-30T14:19:57.565789+00:00,customers,99441,5
1,products.csv,.csv,2828882,2.70,2026-09-30T14:19:57.740161+00:00,products,32951,9
2,orders.csv,.csv,17506326,16.70,2026-09-30T14:19:58.181490+00:00,orders,99441,8
3,order_items.csv,.csv,15024227,14.33,2026-09-30T14:19:58.674311+00:00,order_items,112650,7
4,reviews.csv,.csv,14054760,13.40,2026-09-30T14:19:59.058290+00:00,reviews,99224,7
5,payments.csv,.csv,5737249,5.47,2026-09-30T14:19:59.274266+00:00,payments,103886,5


In [11]:
image_metadata = []

for image_path in IMAGE_FILES:

    stat = image_path.stat()

    image_metadata.append({
        "file_name":
            image_path.name,

        "file_type":
            image_path.suffix.lower(),

        "file_size_bytes":
            stat.st_size,

        "file_size_kb":
            round(
                stat.st_size / 1024,
                2
            ),

        "modified_time":
            datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc
            ).isoformat()
    })


image_metadata_df = pd.DataFrame(
    image_metadata
)

image_metadata_df.head()

,file_name,file_type,file_size_bytes,file_size_kb,modified_time
0,product_001.jpg,.jpg,181972,177.71,2026-09-28T02:55:05.868252+00:00
1,product_002.jpg,.jpg,229635,224.25,2026-09-28T02:55:05.871673+00:00
2,product_003.jpg,.jpg,41352,40.38,2026-09-28T02:55:05.872873+00:00
3,product_004.jpg,.jpg,36381,35.53,2026-09-28T02:55:05.872873+00:00
4,product_005.jpg,.jpg,126543,123.58,2026-09-28T02:55:05.874810+00:00


In [12]:
schema_info = []

for name, df in datasets.items():

    for column in df.columns:

        schema_info.append({
            "dataset": name,
            "column": column,
            "dtype":
                str(
                    df[column].dtype
                )
        })


schema_df = pd.DataFrame(
    schema_info
)

schema_df.head(20)

,dataset,column,dtype
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,products,product_id,str
6,products,product_category_name,str
7,products,product_name_lenght,float64
8,products,product_description_lenght,float64
9,products,product_photos_qty,float64


In [13]:
ingestion_logs = []

In [14]:
for name, path in DATASETS.items():

    object_name = (
        f"landing/"
        f"{name}/"
        f"{path.name}"
    )

    try:

        file_size = (
            path.stat().st_size
        )

        client.fput_object(
            BUCKET_NAME,
            object_name,
            str(path),
            content_type="text/csv"
        )

        ingestion_logs.append({
            "run_id":
                RUN_ID,

            "data_type":
                "CSV",

            "dataset":
                name,

            "file":
                path.name,

            "object":
                object_name,

            "status":
                "SUCCESS",

            "size_bytes":
                file_size,

            "size_mb":
                round(
                    file_size
                    / (1024 * 1024),
                    2
                ),

            "timestamp":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })

        print(
            f"✓ CSV SUCCESS: "
            f"{object_name}"
        )

    except Exception as e:

        ingestion_logs.append({
            "run_id":
                RUN_ID,

            "data_type":
                "CSV",

            "dataset":
                name,

            "file":
                path.name,

            "object":
                object_name,

            "status":
                "FAILED",

            "timestamp":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "error":
                str(e)
        })

        print(
            f"✗ CSV FAILED: "
            f"{object_name}"
        )

        print(" ", e)

✓ CSV SUCCESS: landing/customers/customers.csv
✓ CSV SUCCESS: landing/products/products.csv
✓ CSV SUCCESS: landing/orders/orders.csv
✓ CSV SUCCESS: landing/order_items/order_items.csv
✓ CSV SUCCESS: landing/reviews/reviews.csv
✓ CSV SUCCESS: landing/payments/payments.csv


In [15]:
CONTENT_TYPES = {
    ".jpg":
        "image/jpeg",

    ".jpeg":
        "image/jpeg",

    ".png":
        "image/png",

    ".webp":
        "image/webp"
}


for image_path in IMAGE_FILES:

    object_name = (
        f"images/products/"
        f"{image_path.name}"
    )

    try:

        file_size = (
            image_path.stat().st_size
        )

        extension = (
            image_path
            .suffix
            .lower()
        )

        content_type = (
            CONTENT_TYPES.get(
                extension,
                "application/octet-stream"
            )
        )

        client.fput_object(
            BUCKET_NAME,
            object_name,
            str(image_path),
            content_type=content_type
        )

        ingestion_logs.append({
            "run_id":
                RUN_ID,

            "data_type":
                "IMAGE",

            "file":
                image_path.name,

            "object":
                object_name,

            "status":
                "SUCCESS",

            "size_bytes":
                file_size,

            "size_mb":
                round(
                    file_size
                    / (1024 * 1024),
                    2
                ),

            "timestamp":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })

        print(
            f"✓ IMAGE SUCCESS: "
            f"{image_path.name}"
        )

    except Exception as e:

        ingestion_logs.append({
            "run_id":
                RUN_ID,

            "data_type":
                "IMAGE",

            "file":
                image_path.name,

            "object":
                object_name,

            "status":
                "FAILED",

            "timestamp":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "error":
                str(e)
        })

        print(
            f"✗ IMAGE FAILED: "
            f"{image_path.name}"
        )

        print(" ", e)

✓ IMAGE SUCCESS: product_001.jpg
✓ IMAGE SUCCESS: product_002.jpg
✓ IMAGE SUCCESS: product_003.jpg
✓ IMAGE SUCCESS: product_004.jpg
✓ IMAGE SUCCESS: product_005.jpg


In [16]:
landing_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="landing/",
        recursive=True
    )
)

print(
    f"Số object trong landing/: "
    f"{len(landing_objects)}"
)

for obj in landing_objects:

    print(
        obj.object_name,
        "|",
        obj.size,
        "bytes"
    )

Số object trong landing/: 6
landing/customers/customers.csv | 8661712 bytes
landing/order_items/order_items.csv | 15024227 bytes
landing/orders/orders.csv | 17506326 bytes
landing/payments/payments.csv | 5737249 bytes
landing/products/products.csv | 2828882 bytes
landing/reviews/reviews.csv | 14054760 bytes


In [20]:
image_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="images/products/",
        recursive=True
    )
)

print(
    f"Số image object: "
    f"{len(image_objects):,}"
)

Số image object: 5


In [21]:
ingestion_log_df = pd.DataFrame(
    ingestion_logs
)

ingestion_log_df

,run_id,data_type,dataset,file,object,status,size_bytes,size_mb,timestamp
0,20260930T153809Z,CSV,customers,customers.csv,landing/customers/customers.csv,SUCCESS,8661712,8.26,2026-09-30T15:38:11.107856+00:00
1,20260930T153809Z,CSV,products,products.csv,landing/products/products.csv,SUCCESS,2828882,2.70,2026-09-30T15:38:11.141849+00:00
2,20260930T153809Z,CSV,orders,orders.csv,landing/orders/orders.csv,SUCCESS,17506326,16.70,2026-09-30T15:38:11.306942+00:00
3,20260930T153809Z,CSV,order_items,order_items.csv,landing/order_items/order_items.csv,SUCCESS,15024227,14.33,2026-09-30T15:38:11.431117+00:00
4,20260930T153809Z,CSV,reviews,reviews.csv,landing/reviews/reviews.csv,SUCCESS,14054760,13.40,2026-09-30T15:38:11.542793+00:00
5,20260930T153809Z,CSV,payments,payments.csv,landing/payments/payments.csv,SUCCESS,5737249,5.47,2026-09-30T15:38:11.671194+00:00
6,20260930T153809Z,IMAGE,NaN,product_001.jpg,images/products/product_001.jpg,SUCCESS,181972,0.17,2026-09-30T15:38:11.694202+00:00
7,20260930T153809Z,IMAGE,NaN,product_002.jpg,images/products/product_002.jpg,SUCCESS,229635,0.22,2026-09-30T15:38:11.745834+00:00
8,20260930T153809Z,IMAGE,NaN,product_003.jpg,images/products/product_003.jpg,SUCCESS,41352,0.04,2026-09-30T15:38:11.793142+00:00
9,20260930T153809Z,IMAGE,NaN,product_004.jpg,images/products/product_004.jpg,SUCCESS,36381,0.03,2026-09-30T15:38:11.840878+00:00


In [22]:
log_data = {
    "pipeline":
        "ecommerce-data-lakehouse",

    "process":
        "data_ingestion",

    "run_id":
        RUN_ID,

    "timestamp":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "bucket":
        BUCKET_NAME,

    "summary": {
        "csv_files":
            len(DATASETS),

        "image_files":
            len(IMAGE_FILES),

        "total_records":
            len(ingestion_logs)
    },

    "records":
        ingestion_logs
}


LOG_DIR = (
    BASE_DIR
    / "data"
    / "logs"
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log_file = (
    LOG_DIR
    / f"ingestion_log_{RUN_ID}.json"
)


with open(
    log_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        log_data,
        f,
        indent=4,
        ensure_ascii=False
    )


print(
    f"✓ Đã tạo log local: "
    f"{log_file}"
)

✓ Đã tạo log local: D:\E-commerce-Data-Lakehouse\data\logs\ingestion_log_20260930T153809Z.json


In [23]:
log_object_name = (
    f"logs/ingestion/"
    f"ingestion_log_{RUN_ID}.json"
)

client.fput_object(
    BUCKET_NAME,
    log_object_name,
    str(log_file),
    content_type="application/json"
)

print(
    f"✓ Đã upload log: "
    f"{log_object_name}"
)

✓ Đã upload log: logs/ingestion/ingestion_log_20260930T153809Z.json


In [ ]:
log_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="logs/ingestion/",
        recursive=True
    )
)

print(
    f"Số ingestion log: "
    f"{len(log_objects)}"
)

for obj in log_objects[-10:]:

    print(
        "-",
        obj.object_name
    )

In [ ]:
log_object_name = (
    f"logs/ingestion/"
    f"ingestion_log_{RUN_ID}.json"
)

client.fput_object(
    BUCKET_NAME,
    log_object_name,
    str(log_file),
    content_type="application/json"
)

print(
    f"✓ Đã upload log: "
    f"{log_object_name}"
)

In [ ]:
log_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="logs/ingestion/",
        recursive=True
    )
)

print(
    f"Số ingestion log: "
    f"{len(log_objects)}"
)

for obj in log_objects[-10:]:

    print(
        "-",
        obj.object_name
    )

In [24]:
csv_logs = [
    log
    for log in ingestion_logs
    if log["data_type"] == "CSV"
]

image_logs = [
    log
    for log in ingestion_logs
    if log["data_type"] == "IMAGE"
]


csv_success = sum(
    log["status"] == "SUCCESS"
    for log in csv_logs
)

csv_failed = sum(
    log["status"] == "FAILED"
    for log in csv_logs
)

image_success = sum(
    log["status"] == "SUCCESS"
    for log in image_logs
)

image_failed = sum(
    log["status"] == "FAILED"
    for log in image_logs
)


pipeline_status = (
    "SUCCESS"
    if (
        csv_failed == 0
        and image_failed == 0
    )
    else "PARTIAL_FAILED"
)


print(
    "========== INGESTION SUMMARY =========="
)

print(
    "Run ID:",
    RUN_ID
)

print(
    "Bucket:",
    BUCKET_NAME
)


print("\n--- CSV ---")

print(
    "Expected :",
    len(DATASETS)
)

print(
    "Success  :",
    csv_success
)

print(
    "Failed   :",
    csv_failed
)

print(
    "MinIO    :",
    len(landing_objects)
)


print("\n--- IMAGES ---")

print(
    "Expected :",
    len(IMAGE_FILES)
)

print(
    "Success  :",
    image_success
)

print(
    "Failed   :",
    image_failed
)

print(
    "MinIO    :",
    len(image_objects)
)


print(
    "\nPipeline status:",
    pipeline_status
)


========== INGESTION SUMMARY ==========
Run ID: 20260930T153809Z
Bucket: ecommerce-lakehouse

--- CSV ---
Expected : 6
Success  : 6
Failed   : 0
MinIO    : 6

--- IMAGES ---
Expected : 5
Success  : 5
Failed   : 0
MinIO    : 5

Pipeline status: SUCCESS
